# Live tables

A random-walk ticker appends trades in the background; the session runs a cycle as soon as trades arrive, and each displayed table re-renders when a cycle changes it.

In [1]:
import random
import threading
import time

import kermodes as km
import pyarrow as pa

s = km.local()
ticks = s.input_table(
    {
        "sym": km.string(),
        "price": km.float64(),
        "size": km.int64(),
        "ts": km.timestamp(),
    }
)
stats = ticks.agg_by(
    "sym",
    n="count(*)",
    avg="mean(price)",
    vol="std(price)",
    low="min(price)",
    high="max(price)",
)
latest = ticks.where("size > 100").update("notional = price * size").last_by("sym")

In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META", "TSLA", "AVGO", "JPM", "V"]
prices = dict.fromkeys(SYMBOLS, 100.0)
producing = threading.Event()


def produce() -> None:
    """Appends 200 trades every 50 ms until `producing` is cleared."""
    while producing.is_set():
        syms = random.choices(SYMBOLS, k=200)
        price = []
        for sym in syms:
            prices[sym] = max(0.01, prices[sym] + random.gauss(0, 0.05))
            price.append(round(prices[sym], 2))
        now = time.time_ns()
        ticks.append(
            pa.table(
                {
                    "sym": syms,
                    "price": price,
                    "size": [random.randint(1, 500) for _ in syms],
                    "ts": pa.array([now] * len(syms), pa.timestamp("ns", tz="UTC")),
                }
            )
        )
        time.sleep(0.05)


producing.set()
threading.Thread(target=produce, daemon=True).start()

In [3]:
stats

sym utf8,n int64,avg float64,vol float64,low float64,high float64
AMZN,1317990,140.08669750908447,18.388421009941066,97.84,179.32
TSLA,1317448,119.4528330985358,18.049250528500654,74.4,154.26
GOOG,1318359,65.81376698607824,15.161861380129782,30.64,104.17
AVGO,1319554,122.66584847607464,24.01912301103965,72.76,173.18
V,1318601,94.09020685560004,30.859655302311992,27.94,151.48
MSFT,1318532,155.31452250684984,33.98937578218022,96.83,221.18
JPM,1320780,86.12288237253885,8.244827316484129,59.74,110.46
NVDA,1318504,115.9151652858106,17.195610449944407,75.52,151.41
META,1318725,99.83105476122762,12.202178600057483,75.95,137.2
AAPL,1319507,143.72273746179425,27.358632028245694,85.0,190.63


In [4]:
latest

sym utf8,price float64,size int64,ts timestamp,notional float64
V,29.89,232,2026-11-11 14:25:31.228009,6934.4800000000005
NVDA,98.54,439,2026-11-11 14:25:31.228009,43259.060000000005
TSLA,151.49,399,2026-11-11 14:25:31.228009,60444.51
AMZN,131.43,303,2026-11-11 14:25:31.228009,39823.29
AAPL,175.62,470,2026-11-11 14:25:31.228009,82541.40000000001
META,80.94,410,2026-11-11 14:25:31.228009,33185.4
AVGO,78.84,324,2026-11-11 14:25:31.228009,25544.16
JPM,60.71,386,2026-11-11 14:25:31.228009,23434.06
MSFT,194.64,308,2026-11-11 14:25:31.228009,59949.119999999995
GOOG,66.55,244,2026-11-11 14:25:31.228009,16238.199999999999


In [ ]:
producing.clear()  # stop the ticker; `s.stop()` stops the cycles